# RAASTA — M2 RETRAIN (pothole / crack / speed_bump)

**Goal:** better phone detections (fewer false alarms, higher confidence).

## Before you run
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Free [Roboflow](https://roboflow.com) → copy **API key**
3. Paste key below → run cells **one by one** (or Run all)
4. When done: download `raasta_m2.tflite` → replace  
   `E:\\Android\\projects\\raasta_app\\assets\\models\\raasta_m2.tflite`

This notebook trains **Module 2 only** (3 classes) — same as your Flutter app.

In [ ]:
#@title 1) Roboflow key + install + Drive
ROBOFLOW_API_KEY = ""  #@param {type:"string"}
assert ROBOFLOW_API_KEY.strip(), "Paste your Roboflow API key first"

!pip -q uninstall -y pillow pillow-simd 2>/dev/null
!pip -q install -U "pillow==11.2.1" "ultralytics>=8.3.0" roboflow opencv-python-headless pyyaml

from google.colab import drive
drive.mount("/content/drive")

import os, shutil, random, yaml
from pathlib import Path
from collections import Counter

import torch
from ultralytics import YOLO

DRIVE_DIR = Path("/content/drive/MyDrive/RAASTA_models")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/raasta_m2")
WORK.mkdir(parents=True, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Enable T4 GPU first"

In [ ]:
#@title 2) Class map (M2 only — matches Flutter)
CLASSES = ["pothole", "crack", "speed_bump"]
NAME_TO_ID = {n: i for i, n in enumerate(CLASSES)}

ALIASES = {
    "pothole": "pothole", "potholes": "pothole", "hole": "pothole",
    "pot hole": "pothole", "pot-hole": "pothole",
    "crack": "crack", "cracks": "crack", "longitudinal crack": "crack",
    "transverse crack": "crack", "alligator crack": "crack",
    "road degradation": "crack", "degradation": "crack",
    "speed_bump": "speed_bump", "speed-bump": "speed_bump", "speedbump": "speed_bump",
    "speed bump": "speed_bump", "speed breaker": "speed_bump", "speed-breaker": "speed_bump",
    "bump": "speed_bump", "hump": "speed_bump", "speed hump": "speed_bump",
    "unmarked bump": "speed_bump",
}

def canon(name: str):
    n = name.strip().lower().replace("_", " ")
    n = ALIASES.get(n) or ALIASES.get(n.replace(" ", "_")) or ALIASES.get(name.strip().lower())
    return n if n in NAME_TO_ID else None

print("Classes:", CLASSES)

In [ ]:
#@title 3) Download MORE M2 Roboflow sets (skip M5)
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY.strip())
RAW = WORK / "raw"
RAW.mkdir(exist_ok=True)
sources = []

def download_rf(workspace, project, version, folder_name):
    out = RAW / folder_name
    if out.exists() and any(out.rglob("*.jpg")):
        print("[skip exists]", folder_name)
        return out
    print("[download]", workspace, project, version)
    ds = rf.workspace(workspace).project(project).version(version).download(
        "yolov8", location=str(out)
    )
    return Path(ds.location)

# Extra pothole-focused sets + your previous three
jobs = [
    ("speed-bump-detection", "speed-bump-detection-se0eh", 15, "speed_bump_v15"),
    ("nsip-project", "road-degradation-beta", 1, "road_degradation"),
    ("pothole-detection-1nczj", "speed-unmarked-bumb", 1, "unmarked_bump"),
    # More potholes (public Universe — skip quietly if a version 404s)
    ("brad-dwyer", "pothole-voxrl", 1, "pothole_brad"),
    ("project-jnlc6", "pothole-detection-yolov8", 1, "pothole_yolov8"),
]

for ws, proj, ver, name in jobs:
    try:
        sources.append(download_rf(ws, proj, ver, name))
    except Exception as e:
        print("[warn skip]", name, e)

print("Downloaded:", [p.name for p in sources])
assert sources, "No datasets downloaded — check API key / internet"

In [ ]:
#@title 4) Merge into one YOLO dataset (3 classes)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MERGED = WORK / "merged"
for sub in ("images/train", "images/val", "labels/train", "labels/val"):
    p = MERGED / sub
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True)

def load_names(root: Path):
    for fname in ("data.yaml", "dataset.yaml"):
        yml = root / fname
        if yml.is_file():
            data = yaml.safe_load(yml.read_text())
            names = data.get("names")
            if isinstance(names, dict):
                return [names[k] for k in sorted(names, key=lambda x: int(x))]
            if isinstance(names, list):
                return names
    return None

def find_pairs(root: Path):
    items = []
    for split in ("train", "val", "valid", "test"):
        for img_dir_name in (f"images/{split}", split, f"{split}/images"):
            img_dir = root / img_dir_name
            if not img_dir.is_dir():
                continue
            lbl_dir = root / img_dir_name.replace("images", "labels")
            if not lbl_dir.is_dir():
                lbl_dir = root / ("valid" if split == "valid" else split) / "labels"
            if not lbl_dir.is_dir():
                lbl_dir = root / "labels" / ("val" if split in ("val", "valid") else split)
            for img in img_dir.rglob("*"):
                if img.suffix.lower() not in IMG_EXTS:
                    continue
                lbl = lbl_dir / f"{img.stem}.txt"
                if not lbl.is_file():
                    alt = list(root.rglob(f"{img.stem}.txt"))
                    lbl = alt[0] if alt else None
                key = "val" if split in ("val", "valid", "test") else "train"
                items.append((key, img, lbl))
            break
    return items

counts = Counter()
n_ok = 0
for src in sources:
    names = load_names(src) or []
    print("[merge]", src.name, names)
    id_map = {}
    for i, nm in enumerate(names):
        c = canon(str(nm))
        if c:
            id_map[i] = NAME_TO_ID[c]
    if not id_map:
        print("  (no mappable classes — skip)")
        continue
    for split, img, lbl in find_pairs(src):
        if lbl is None or not Path(lbl).is_file():
            continue
        lines_out = []
        for line in Path(lbl).read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            old = int(float(parts[0]))
            if old not in id_map:
                continue
            new_id = id_map[old]
            lines_out.append(" ".join([str(new_id)] + parts[1:5]))
            counts[CLASSES[new_id]] += 1
        if not lines_out:
            continue
        stem = f"{src.name}_{img.stem}_{n_ok}"
        shutil.copy2(img, MERGED / "images" / split / f"{stem}{img.suffix.lower()}")
        (MERGED / "labels" / split / f"{stem}.txt").write_text("\n".join(lines_out) + "\n")
        n_ok += 1

data_yaml = {
    "path": str(MERGED),
    "train": "images/train",
    "val": "images/val",
    "names": {i: n for i, n in enumerate(CLASSES)},
}
(MERGED / "data.yaml").write_text(yaml.safe_dump(data_yaml))
print("Images kept:", n_ok)
print("Label counts:", dict(counts))
assert n_ok > 200, "Too few images — check downloads"

In [ ]:
#@title 5) Train / fine-tune (M2)
# Prefer continuing from your last good weights on Drive.
prev = DRIVE_DIR / "raasta_m2_best.pt"
if not prev.is_file():
    prev = DRIVE_DIR / "raasta_m2_m5.pt"

if prev.is_file():
    MODEL = str(prev)
    print("Fine-tuning from:", MODEL)
else:
    MODEL = "yolov8n.pt"
    print("Starting fresh from yolov8n.pt")

EPOCHS = 80
IMGSZ = 416
BATCH = 8  # drop to 4 if Colab OOMs

model = YOLO(MODEL)
model.train(
    data=str(MERGED / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=2,
    project=str(WORK / "runs"),
    name="m2_retrain",
    exist_ok=True,
    patience=20,
    cos_lr=True,
    close_mosaic=15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    plots=False,
)

best = Path(model.trainer.best)
print("Best weights:", best)
metrics = model.val(data=str(MERGED / "data.yaml"), imgsz=IMGSZ, batch=BATCH, plots=False)
print("mAP50:", float(metrics.box.map50))
print("Per-class mAP50:", [float(x) for x in metrics.box.ap50])

In [ ]:
#@title 6) Export TFLite + save to Drive + download
from google.colab import files

best = Path(WORK / "runs" / "m2_retrain" / "weights" / "best.pt")
assert best.is_file(), "Train cell must finish first"

shutil.copy2(best, DRIVE_DIR / "raasta_m2_best.pt")

export_model = YOLO(str(best))
# 320 matches Flutter YoloM2Interpreter.inputSize
export_path = export_model.export(format="tflite", imgsz=320)
tflite = Path(export_path)
if not tflite.suffix == ".tflite":
    cands = list(tflite.parent.glob("*.tflite"))
    tflite = cands[0] if cands else tflite

out_tflite = DRIVE_DIR / "raasta_m2.tflite"
shutil.copy2(tflite, out_tflite)
local_copy = WORK / "raasta_m2.tflite"
shutil.copy2(tflite, local_copy)

print("Saved Drive:", out_tflite)
print("Size MB:", round(out_tflite.stat().st_size / 1e6, 2))
files.download(str(local_copy))
print("\\nNext on PC: replace assets/models/raasta_m2.tflite then flutter run")

### Done — success check

| Metric | Target for FYP |
|--------|----------------|
| Overall mAP50 | **≥ 0.55** (good), ≥ 0.65 (strong) |
| speed_bump | ≥ 0.75 |
| pothole | ≥ 0.45 |
| crack | ≥ 0.30 |

If pothole still &lt; 0.40: add another Roboflow pothole dataset and run cells 3→6 again (fine-tune continues from Drive `raasta_m2_best.pt`).

**Phone:** full restart after replacing the `.tflite` file.